In [1]:
import pandas as pd
import os
from os.path import dirname


root_path = dirname(os.getcwd())

pd.set_option("display.max_columns", None)
data_dir = root_path + "/data/datasets/original/"
data_dir_processed = root_path + "/data/datasets/processed/"
data_dir_graphs = root_path + "/data/datasets/graphs_repair/"

print(root_path, data_dir, data_dir_processed, data_dir_graphs, sep="\n")

/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/original/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/processed/
/home/matteo/Documents/GNN-test2/SEPH_MODELS/SEPH_OUTCOME/data/datasets/graphs_repair/


In [2]:
dataset = "bpic2012_O_DECLINED-COMPLETE" # Select dataset to process

In [3]:
#if dataset == "bpic2012_O_DECLINED":
#    raw_data = pd.read_csv(f"datasets/original/{dataset}.csv")
#else:
raw_data = pd.read_csv(f"datasets/original/{dataset}.csv", sep=";")

raw_data.head()

,AMOUNT_REQ,Case ID,label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,month,weekday,hour,open_cases,Complete Timestamp
0,20000,173688,regular,A_SUBMITTED-COMPLETE,112.0,COMPLETE,98,0.000000,0.000000,1,10,5,1,1,2011-10-01 01:38:44.546
1,20000,173688,regular,A_PARTLYSUBMITTED-COMPLETE,112.0,COMPLETE,98,0.005567,0.005567,2,10,5,1,1,2011-10-01 01:38:44.880
2,20000,173688,regular,A_PREACCEPTED-COMPLETE,112.0,COMPLETE,99,0.883767,0.889333,3,10,5,1,1,2011-10-01 01:39:37.906
3,20000,173688,regular,W_Completeren aanvraag-SCHEDULE,112.0,SCHEDULE,99,0.016150,0.905483,4,10,5,1,1,2011-10-01 01:39:38.875
4,20000,173688,regular,W_Completeren aanvraag-START,112.0,START,756,657.126033,658.031517,5,10,5,12,10,2011-10-01 12:36:46.437


In [4]:
if dataset == "BPIC11_f1" :
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID", "Activity code": "Activity","label":"Label"}
    )
elif dataset == "BPIC15_3_f2" or dataset == "sepsis_cases_2" or dataset == "BPIC17_O_Cancelled":
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID","label":"Label"}
    )
elif dataset == "bpic2012_O_DECLINED-COMPLETE" or dataset == "traffic_fines_1" or dataset == "hospital_billing_3":
    tab_all = raw_data.rename(
        columns={"Case ID": "CaseID","label":"Label","Complete Timestamp":"time:timestamp"}
    )
else: 
    raise ValueError(f"Unknown dataset name: {dataset!r}")

In [5]:
from datetime import datetime
import time

def translate_time(time_str):
    return datetime.fromisoformat(time_str).timestamp()

In [6]:
tab_all["time:timestamp"] = tab_all["time:timestamp"].apply(translate_time)

In [7]:
tab_all.head()

,AMOUNT_REQ,CaseID,Label,Activity,Resource,lifecycle:transition,timesincemidnight,timesincelastevent,timesincecasestart,event_nr,month,weekday,hour,open_cases,time:timestamp
0,20000,173688,regular,A_SUBMITTED-COMPLETE,112.0,COMPLETE,98,0.000000,0.000000,1,10,5,1,1,1.317419e+09
1,20000,173688,regular,A_PARTLYSUBMITTED-COMPLETE,112.0,COMPLETE,98,0.005567,0.005567,2,10,5,1,1,1.317419e+09
2,20000,173688,regular,A_PREACCEPTED-COMPLETE,112.0,COMPLETE,99,0.883767,0.889333,3,10,5,1,1,1.317419e+09
3,20000,173688,regular,W_Completeren aanvraag-SCHEDULE,112.0,SCHEDULE,99,0.016150,0.905483,4,10,5,1,1,1.317419e+09
4,20000,173688,regular,W_Completeren aanvraag-START,112.0,START,756,657.126033,658.031517,5,10,5,12,10,1.317458e+09


In [8]:
split_ratio = 4 / 5

first_act_tab = (
    tab_all.groupby("CaseID").first().sort_values("time:timestamp").reset_index()
)
first_act_tab = first_act_tab[
    ~first_act_tab.duplicated(subset=["CaseID", "Activity"], keep="first")
]
first_act_tab = first_act_tab.reset_index(drop=True)

list_train_valid_cases = list(
    first_act_tab[: int(split_ratio * len(first_act_tab))]["CaseID"].unique()
)

list_train_cases = list_train_valid_cases[: int(len(list_train_valid_cases) * 0.8)]
tab_train = tab_all[tab_all["CaseID"].isin(list_train_cases)].reset_index(drop=True)

list_valid_cases = list_train_valid_cases[int(len(list_train_valid_cases) * 0.8) :]
tab_valid = tab_all[tab_all["CaseID"].isin(list_valid_cases)].reset_index(drop=True)

list_test_cases = list(
    first_act_tab[int(split_ratio * len(first_act_tab)) :]["CaseID"].unique()
)
tab_test = tab_all[tab_all["CaseID"].isin(list_test_cases)].reset_index(drop=True)


#Find earliest timestamp in testing set.
split_ts = tab_test["time:timestamp"].min()

#Remove elements
tab_train = tab_train[tab_train["time:timestamp"] < split_ts].reset_index(drop=True)
tab_valid = tab_valid[tab_valid["time:timestamp"] < split_ts].reset_index(drop=True)




In [9]:
tab_all.to_csv(data_dir_processed + f"{dataset}_processed_all.csv", index=False)

In [10]:
tab_train.to_csv(data_dir_processed+ f"{dataset}_processed_train.csv", index = False)

In [11]:
tab_valid.to_csv(data_dir_processed+f"{dataset}_processed_valid.csv", index = False)

In [12]:
tab_test.to_csv(data_dir_processed+ f"{dataset}_processed_test.csv", index = False)